# 05. Дизайн A/B-теста и валидация через симуляцию

В ноутбуке 03 сформулирована гипотеза о том, что триггерная кампания реактивации увеличивает retention в сегменте клиентов с низким первым чеком. В этом ноутбуке выполняется полный дизайн A/B-теста, необходимый для проверки гипотезы в продуктовой среде.

Структура ноутбука:
1. Расчёт базовой конверсии в целевом сегменте (исходное состояние H0).
2. Формализация альтернативной гипотезы: базовая конверсия плюс минимально детектируемый эффект.
3. Расчёт необходимого размера выборки на каждую группу по формуле для двух пропорций.
4. Построение power curve для оценки чувствительности дизайна к величине эффекта.
5. Монте-Карло симуляция для эмпирической проверки заявленной мощности.
6. Проверка контроля ошибок первого рода через симуляцию при отсутствии эффекта.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
rng = np.random.default_rng(42)

customers = pd.read_parquet('../data/customers_labeled.parquet')
print(f'Клиентов в данных: {len(customers):,}')

## Базовая конверсия в целевом сегменте

Конверсия определяется как доля клиентов, совершивших как минимум одну дополнительную покупку после первой. Базовая величина рассчитывается по сегменту «низкий первый чек», поскольку именно он является целевой аудиторией предполагаемой кампании.

In [ ]:
low = customers[customers['check_segment'].str.startswith('Низкий')]
p_baseline = low['returned'].mean()
print(f'Размер сегмента: {len(low):,} клиентов')
print(f'Базовая доля возврата (p_baseline): {p_baseline*100:.2f}%')

## Параметры теста

Параметры фиксируются до проведения симуляции, чтобы исключить ретроспективную подгонку.

- Уровень значимости alpha = 0.05. Тест двусторонний: эффект потенциально может быть как положительным, так и отрицательным (например, если кампания вызывает усталость от коммуникаций).
- Целевая мощность 1 - beta = 0.80. Стандартная величина для продуктовых экспериментов.
- Минимально детектируемый эффект MDE = 2 процентных пункта в абсолюте. Меньшие эффекты экономически нецелесообразны: предельная стоимость кампании на единицу прироста retention не окупится.

In [ ]:
ALPHA = 0.05
POWER = 0.80
MDE = 0.02  # 2 п.п. в абсолюте

p_test = p_baseline + MDE
print(f'p_baseline (контроль):  {p_baseline*100:.2f}%')
print(f'p_test     (тест):      {p_test*100:.2f}%')
print(f'абсолютный MDE:         {MDE*100:.1f} п.п.')
print(f'относительный лифт:     {MDE/p_baseline*100:.1f}%')

## Расчёт размера выборки

Используется аппарат `statsmodels` для проверки гипотезы о равенстве двух пропорций. В основе лежит стандартная формула на квантилях нормального распределения; готовая реализация выбрана для прозрачности и воспроизводимости.

In [ ]:
effect_size = proportion_effectsize(p_test, p_baseline)
analysis = NormalIndPower()
n_per_group = analysis.solve_power(
    effect_size=effect_size,
    alpha=ALPHA,
    power=POWER,
    ratio=1.0,
    alternative='two-sided',
)
n_per_group = int(np.ceil(n_per_group))
n_total = n_per_group * 2

print(f'Effect size (Cohen h):       {effect_size:.4f}')
print(f'Требуется на одну группу:    {n_per_group:,} клиентов')
print(f'Всего на тест:               {n_total:,} клиентов')
print(f'Размер сегмента в данных:    {len(low):,} клиентов')
print()
print(f'Объёма данных достаточно для одного тестового цикла: {"да" if len(low) >= n_total else "нет"}')

Приведённый размер выборки задаёт операционные параметры теста. При фиксированном объёме поступающих клиентов в целевом сегменте можно дополнительно оценить плановую длительность эксперимента; этот расчёт зависит от продуктовых данных, не входящих в датасет.

## Power curve

Зависимость требуемого размера выборки от величины MDE существенна для разговора с продуктовой командой. Уменьшение MDE приводит к нелинейному росту необходимой выборки, что напрямую влияет на длительность теста и стоимость задержки решения.

In [ ]:
mde_grid = np.linspace(0.005, 0.05, 30)
n_required = []
for mde in mde_grid:
    es = proportion_effectsize(p_baseline + mde, p_baseline)
    n = analysis.solve_power(effect_size=es, alpha=ALPHA, power=POWER, ratio=1.0, alternative='two-sided')
    n_required.append(int(np.ceil(n)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mde_grid * 100, n_required, color='#4C72B0', linewidth=2)
ax.axvline(x=MDE * 100, color='red', linestyle='--', alpha=0.6, label=f'выбранный MDE = {MDE*100:.0f} п.п.')
ax.set_yscale('log')
ax.set_xlabel('MDE, абсолютные процентные пункты')
ax.set_ylabel('Размер одной группы (логарифмическая шкала)')
ax.set_title('Зависимость требуемого размера выборки от MDE')
ax.legend()
plt.tight_layout()
plt.savefig('../images/power_curve.png', dpi=120, bbox_inches='tight')
plt.show()

Логарифмическая шкала по вертикальной оси выбрана осмысленно: при уменьшении MDE с 2 п.п. до 1 п.п. требуемый размер выборки увеличивается приблизительно в четыре раза. Эта нелинейность объясняет распространённый компромисс в продуктовых экспериментах: MDE = 2 п.п. позволяет завершить тест в разумные сроки при сохранении приемлемой чувствительности.

## Монте-Карло валидация дизайна

Аналитический расчёт мощности проверяется эмпирически. На рассчитанном размере выборки запускается серия из 2000 виртуальных A/B-тестов с известным истинным эффектом. Доля тестов, в которых эффект корректно идентифицирован как значимый, должна совпадать с заявленной мощностью.

In [ ]:
N_SIMULATIONS = 2000
p_control = p_baseline
p_treatment = p_baseline + MDE

significant_results = 0
for _ in range(N_SIMULATIONS):
    control_outcomes = rng.binomial(1, p_control, size=n_per_group)
    treat_outcomes = rng.binomial(1, p_treatment, size=n_per_group)
    n1, n2 = n_per_group, n_per_group
    p1 = control_outcomes.mean()
    p2 = treat_outcomes.mean()
    p_pool = (control_outcomes.sum() + treat_outcomes.sum()) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z = (p2 - p1) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    if p_value < ALPHA:
        significant_results += 1

empirical_power = significant_results / N_SIMULATIONS
print(f'Запущено симуляций:           {N_SIMULATIONS:,}')
print(f'Значимых результатов:         {significant_results:,}')
print(f'Эмпирическая мощность:        {empirical_power*100:.1f}%')
print(f'Расчётная мощность (целевая): {POWER*100:.0f}%')

Эмпирическая мощность ожидаемо находится в окрестности 80%, что подтверждает корректность аналитического расчёта. При заданном размере выборки тест действительно обнаруживает истинный эффект 2 п.п. примерно в четырёх случаях из пяти.

## Контроль ошибок первого рода

Симметричная проверка дизайна: симулируется ситуация, в которой обе группы имеют одинаковую истинную конверсию. Доля тестов, в которых регистрируется значимое различие, должна соответствовать выбранному уровню alpha. Существенное превышение указывает на проблему в дизайне.

In [ ]:
false_positives = 0
for _ in range(N_SIMULATIONS):
    control_outcomes = rng.binomial(1, p_baseline, size=n_per_group)
    treat_outcomes = rng.binomial(1, p_baseline, size=n_per_group)
    n1, n2 = n_per_group, n_per_group
    p1 = control_outcomes.mean()
    p2 = treat_outcomes.mean()
    p_pool = (control_outcomes.sum() + treat_outcomes.sum()) / (n1 + n2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
    z = (p2 - p1) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    if p_value < ALPHA:
        false_positives += 1

fpr = false_positives / N_SIMULATIONS
print(f'Эмпирическая доля ложноположительных результатов: {fpr*100:.1f}%')
print(f'Целевое значение alpha: {ALPHA*100:.0f}%')

Доля ложноположительных результатов соответствует заданному уровню alpha = 5%. Это означает, что дизайн корректно контролирует ошибку первого рода и не выдаёт значимый результат чаще, чем в одном случае из двадцати при отсутствии истинного эффекта.

## Сводный протокол теста

Параметры эксперимента, фиксируемые до запуска:
- Случайное распределение клиентов по группам в момент совершения первой транзакции в сегменте «низкий первый чек».
- Основная метрика: доля клиентов, совершивших вторую покупку в течение 30 дней после первой.
- alpha = 5%, мощность = 80%, MDE = 2 процентных пункта в абсолюте.
- Размер каждой группы рассчитан выше и подтверждён симуляцией.
- Тест останавливается строго по достижении плановых размеров групп; промежуточный анализ значимости (peeking) запрещён.

Контрольные (guardrail) метрики, отслеживаемые параллельно:
- Средний размер второй покупки.
- Совокупная выручка с клиента в течение 60 дней после первой транзакции.

Без отслеживания guardrail-метрик возможен сценарий, при котором рост retention сопровождается снижением среднего чека повторных покупок и нулевым или отрицательным итоговым эффектом в денежном выражении.